### Step 0 - Build Unified Metadata

**Goal:** Construct a unified DataFrame from the ISIC 2019 CVs (GroundTruth + Metadata), derive the source of each image based on the `lesion_id` prefix, apply the mapping **AK + SCC -> akiec** for compatibility with HAM10000, and produce `unified_metadata.csv`.

**Output:**
*   `unified_metadata.csv` - Unified schema, ready for splitting.
*   `step0_report.md` - Report containing statistics and sanity checks.

**Next Step:** STEP 1 (Lesion-level split and construction of the fixed test set).

## 1. Setup - Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
IN_COLAB = True


Mounted at /content/drive


In [2]:
BASE_DIR = '/content/drive/MyDrive/Thesis'

GROUNDTRUTH_PATH = f'{BASE_DIR}/data/ISIC2019/ISIC_2019_Training_GroundTruth.csv'
METADATA_PATH    = f'{BASE_DIR}/data/ISIC2019/ISIC_2019_Training_Metadata.csv'
IMAGES_DIR       = f'{BASE_DIR}/data/ISIC2019/ISIC_2019_Training_Input/'
OUTPUT_DIR       = f'{BASE_DIR}/pipeline_outputs/step0'

import os
for p in [GROUNDTRUTH_PATH, METADATA_PATH, IMAGES_DIR]:
    exists = os.path.exists(p)
    print(f"{'[OK]' if exists else '[MISSING]'}  {p}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\nOutput dir: {OUTPUT_DIR}")

[OK]  /content/drive/MyDrive/Thesis/data/ISIC2019/ISIC_2019_Training_GroundTruth.csv
[OK]  /content/drive/MyDrive/Thesis/data/ISIC2019/ISIC_2019_Training_Metadata.csv
[OK]  /content/drive/MyDrive/Thesis/data/ISIC2019/ISIC_2019_Training_Input/

Output dir: /content/drive/MyDrive/Thesis/pipeline_outputs/step0


## 2. Imports

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

## 3. Configuration Constants

**Mapping ISIC 2019 -> HAM10000-compatible:**
*   **AK -> akiec** (Actinic Keratosis = part of the original AKIEC)
*   **SCC -> akiec** (**MERGED** — original HAM10000 convention: AKIEC = AK + IEC)
*   **UNK -> excluded** (out-of-distribution class, irrelevant for classification)
*   **Remainder -> identity** (lower-case)


In [4]:
GT_LABEL_COLUMNS = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']

ISIC_TO_HAM_MAPPING = {
    'MEL':  'mel',
    'NV':   'nv',
    'BCC':  'bcc',
    'AK':   'akiec',     # AK -> akiec
    'BKL':  'bkl',
    'DF':   'df',
    'VASC': 'vasc',
    'SCC':  'akiec',     # MERGED: SCC + AK = AKIEC
    'UNK':  'unknown',   # excluded
}

In [5]:
def derive_source(lesion_id):
    """
    Derivează sursa imaginii din prefixul lesion_id-ului.

    Convenție ISIC pentru lesion_id-uri:
    - HAM_*    -> HAM10000 (ViDIR Vienna)
    - BCN_*    -> BCN_20000 (Hospital Clinic Barcelona)
    - MSK*     -> Memorial Sloan Kettering (ex: MSK4_0011169)
    - IL_*     -> ISIC Internal/Various
    - alt prefix sau lipsă -> unknown_source
    """
    if pd.isna(lesion_id) or lesion_id == '':
        return 'unknown_source'

    lid = str(lesion_id)
    if lid.startswith('HAM_'):
        return 'HAM10000'
    elif lid.startswith('BCN_'):
        return 'BCN_20000'
    elif lid.startswith('MSK'):
        return 'MSK'
    elif lid.startswith('IL_'):
        return 'ISIC_internal'
    else:
        return 'other'


def onehot_to_label(row, label_columns=GT_LABEL_COLUMNS):
    """Convertește one-hot encoding la single label."""
    for col in label_columns:
        if row[col] == 1.0:
            return col
    return None

## 4. Citire și combinare CSV-uri

In [6]:
print(f"Reading GroundTruth: {GROUNDTRUTH_PATH}")
gt = pd.read_csv(GROUNDTRUTH_PATH)
print(f"  -> {len(gt):,} rows")
print(f"  -> Columns: {gt.columns.tolist()}")
gt.head(3)

Reading GroundTruth: /content/drive/MyDrive/Thesis/data/ISIC2019/ISIC_2019_Training_GroundTruth.csv
  -> 25,331 rows
  -> Columns: ['image', 'MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']


,image,MEL,NV,BCC,AK,BKL,DF,VASC,SCC,UNK
0,ISIC_0000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,ISIC_0000001,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,ISIC_0000002,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
print(f"Reading Metadata: {METADATA_PATH}")
meta = pd.read_csv(METADATA_PATH)
print(f"  -> {len(meta):,} rows")
print(f"  -> Columns: {meta.columns.tolist()}")
meta.head(3)

Reading Metadata: /content/drive/MyDrive/Thesis/data/ISIC2019/ISIC_2019_Training_Metadata.csv
  -> 25,331 rows
  -> Columns: ['image', 'age_approx', 'anatom_site_general', 'lesion_id', 'sex']


,image,age_approx,anatom_site_general,lesion_id,sex
0,ISIC_0000000,55.0,anterior torso,NaN,female
1,ISIC_0000001,30.0,anterior torso,NaN,female
2,ISIC_0000002,60.0,upper extremity,NaN,female


In [8]:
merged = gt.merge(meta, on='image', how='outer', indicator=True)

only_gt   = (merged['_merge'] == 'left_only').sum()
only_meta = (merged['_merge'] == 'right_only').sum()
both      = (merged['_merge'] == 'both').sum()

print(f"Merge results:")
print(f"  În ambele: {both:,}")
print(f"  Doar în GroundTruth (fără metadata): {only_gt:,}")
print(f"  Doar în Metadata (fără labels): {only_meta:,}")

merged = merged[merged['_merge'].isin(['left_only', 'both'])].copy()
merged = merged.drop(columns=['_merge'])
print(f"\nPăstrăm: {len(merged):,} imagini cu labels.")

Merge results:
  În ambele: 25,331
  Doar în GroundTruth (fără metadata): 0
  Doar în Metadata (fără labels): 0

Păstrăm: 25,331 imagini cu labels.


## 5. Construire schema unificată

In [9]:
print("Construiesc schema unificată...")

df = pd.DataFrame()
df['image_id'] = merged['image']

# Convert one-hot -> single ISIC label
isic_label = merged.apply(onehot_to_label, axis=1)
df['dx_isic'] = isic_label
df['dx'] = isic_label.map(ISIC_TO_HAM_MAPPING)

# Lesion_id (poate fi NaN)
df['lesion_id'] = merged['lesion_id']

# Lesion_id sintetic pentru imaginile fără lesion_id
missing_lesion = df['lesion_id'].isna() | (df['lesion_id'] == '')
n_missing = missing_lesion.sum()
if n_missing > 0:
    synthetic_ids = 'SYNTH_' + df.loc[missing_lesion, 'image_id'].astype(str)
    df.loc[missing_lesion, 'lesion_id'] = synthetic_ids
    print(f"  {n_missing:,} imagini fără lesion_id -> au primit ID sintetic 'SYNTH_<image_id>'")

# Derivă sursa din lesion_id (sintetic -> unknown_source)
df['source'] = df['lesion_id'].apply(derive_source)

# Path imagine
df['image_path'] = df['image_id'].apply(lambda x: os.path.join(IMAGES_DIR, f"{x}.jpg"))

# Metadata pacient
df['age']         = merged['age_approx']
df['sex']         = merged['sex']
df['anatom_site'] = merged['anatom_site_general']

print(f"\nDataFrame construit: {len(df):,} rânduri × {len(df.columns)} coloane")
df.head(5)

Construiesc schema unificată...
  2,084 imagini fără lesion_id -> au primit ID sintetic 'SYNTH_<image_id>'

DataFrame construit: 25,331 rânduri × 9 coloane


,image_id,dx_isic,dx,lesion_id,source,image_path,age,sex,anatom_site
0,ISIC_0000000,NV,nv,SYNTH_ISIC_0000000,other,/content/drive/MyDrive/Thesis/data/ISIC2019/IS...,55.0,female,anterior torso
1,ISIC_0000001,NV,nv,SYNTH_ISIC_0000001,other,/content/drive/MyDrive/Thesis/data/ISIC2019/IS...,30.0,female,anterior torso
2,ISIC_0000002,MEL,mel,SYNTH_ISIC_0000002,other,/content/drive/MyDrive/Thesis/data/ISIC2019/IS...,60.0,female,upper extremity
3,ISIC_0000003,NV,nv,SYNTH_ISIC_0000003,other,/content/drive/MyDrive/Thesis/data/ISIC2019/IS...,30.0,male,upper extremity
4,ISIC_0000004,MEL,mel,SYNTH_ISIC_0000004,other,/content/drive/MyDrive/Thesis/data/ISIC2019/IS...,80.0,male,posterior torso


## 6. Excludere clasa UNK (out-of-distribution)

In [10]:
n_before = len(df)
df = df[df['dx'] != 'unknown'].copy()
n_excluded = n_before - len(df)

print(f"Excluse {n_excluded:,} imagini cu dx='unknown' (clasa UNK din ISIC 2019).")
print(f"Imagini rămase: {len(df):,}")

Excluse 0 imagini cu dx='unknown' (clasa UNK din ISIC 2019).
Imagini rămase: 25,331


## 7. Sanity checks

In [11]:
# Check 1: Niciun lesion_id nu apare în multiple surse
lesion_source_count = df.groupby('lesion_id')['source'].nunique()
multi_source = lesion_source_count[lesion_source_count > 1]

if len(multi_source) > 0:
    print(f"[ATENȚIE] {len(multi_source)} lesion_id-uri apar în mai multe surse!")
    print("Exemple:")
    for lid in multi_source.head(5).index:
        sources = df[df['lesion_id'] == lid]['source'].unique()
        print(f"  {lid}: {sources.tolist()}")
else:
    print("[OK] Niciun lesion_id nu apare în două surse diferite.")

[OK] Niciun lesion_id nu apare în două surse diferite.


In [12]:
# Check 2: Path-urile imaginilor există fizic (sample de 200)
print("Verific existența imaginilor (sample de 200)...")

sample = df.sample(min(200, len(df)), random_state=42)
missing = [path for path in sample['image_path'] if not os.path.exists(path)]

if missing:
    print(f"[ATENȚIE] {len(missing)}/200 imagini sample lipsesc!")
    print(f"Primele 5 lipsă: {missing[:5]}")
    print("Verifică că IMAGES_DIR e corect și că extensia e .jpg")
else:
    print("[OK] Toate 200 imaginile sample există pe disc.")

Verific existența imaginilor (sample de 200)...
[OK] Toate 200 imaginile sample există pe disc.


## 8. Statistici de distribuție

In [13]:
# Distribuție per clasă
print("=" * 50)
print("Distribuție per clasă (după AK+SCC merge):")
print("=" * 50)
class_dist = df['dx'].value_counts()
for cls, count in class_dist.items():
    pct = 100 * count / len(df)
    print(f"  {cls:8s}  {count:6,d}  ({pct:5.1f}%)")
print(f"  {'TOTAL':8s}  {len(df):6,d}")

Distribuție per clasă (după AK+SCC merge):
  nv        12,875  ( 50.8%)
  mel        4,522  ( 17.9%)
  bcc        3,323  ( 13.1%)
  bkl        2,624  ( 10.4%)
  akiec      1,495  (  5.9%)
  vasc         253  (  1.0%)
  df           239  (  0.9%)
  TOTAL     25,331


In [14]:
# Distribuție per sursă
print("=" * 60)
print("Distribuție per sursă:")
print("=" * 60)
source_dist = df['source'].value_counts()
for src, count in source_dist.items():
    pct = 100 * count / len(df)
    n_lesions = df[df['source'] == src]['lesion_id'].nunique()
    print(f"  {src:18s}  {count:6,d}  ({pct:5.1f}%)  | {n_lesions:5,d} lesion_id-uri unice")
print(f"  {'TOTAL':18s}  {len(df):6,d}")

Distribuție per sursă:
  BCN_20000           12,413  ( 49.0%)  | 3,576 lesion_id-uri unice
  HAM10000            10,015  ( 39.5%)  | 7,470 lesion_id-uri unice
  other                2,084  (  8.2%)  | 2,084 lesion_id-uri unice
  MSK                    819  (  3.2%)  |   801 lesion_id-uri unice
  TOTAL               25,331


In [15]:
# Cross-tab clasă × sursă
print("=" * 60)
print("Distribuție clasă × sursă:")
print("=" * 60)
cross = pd.crosstab(df['dx'], df['source'], margins=True, margins_name='TOTAL')
cross

Distribuție clasă × sursă:


source,BCN_20000,HAM10000,MSK,other,TOTAL
dx,,,,,
akiec,1168,327,0,0,1495
bcc,2809,514,0,0,3323
bkl,1138,1099,189,198,2624
df,124,115,0,0,239
mel,2857,1113,215,337,4522
nv,4206,6705,415,1549,12875
vasc,111,142,0,0,253
TOTAL,12413,10015,819,2084,25331


In [16]:
print("=" * 50)
print("Disponibilitate metadata pacient:")
print("=" * 50)
for col in ['age', 'sex', 'anatom_site']:
    avail = df[col].notna().sum()
    pct = 100 * avail / len(df)
    print(f"  {col:12s}  {avail:6,d} / {len(df):,}  ({pct:5.1f}%)")

Disponibilitate metadata pacient:
  age           24,894 / 25,331  ( 98.3%)
  sex           24,947 / 25,331  ( 98.5%)
  anatom_site   22,700 / 25,331  ( 89.6%)


## 9. Salvare output-uri

In [17]:
output_csv = os.path.join(OUTPUT_DIR, 'unified_metadata.csv')
df.to_csv(output_csv, index=False)
print(f"[SAVED] {output_csv}")
print(f"  -> {len(df):,} rânduri × {len(df.columns)} coloane")

[SAVED] /content/drive/MyDrive/Thesis/pipeline_outputs/step0/unified_metadata.csv
  -> 25,331 rânduri × 9 coloane


In [18]:
# Generare raport markdown
def generate_report(df, output_path):
    lines = []
    lines.append("# PASUL 0 — Raport unified metadata")
    lines.append("")
    lines.append(f"**Data generării:** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append("")
    lines.append("## 1. Statistici generale")
    lines.append("")
    lines.append(f"- **Total imagini:** {len(df):,}")
    lines.append(f"- **Lesion_id-uri unice:** {df['lesion_id'].nunique():,}")
    lines.append(f"- **Imagini per lesion_id (medie):** {len(df) / df['lesion_id'].nunique():.2f}")
    lines.append(f"- **Clase unice (dx):** {df['dx'].nunique()}")
    lines.append(f"- **Surse unice:** {df['source'].nunique()}")
    lines.append("")

    lines.append("## 2. Distribuție per clasă (după AK+SCC merge)")
    lines.append("")
    lines.append("| Clasă | Imagini | % |")
    lines.append("|-------|---------|---|")
    class_counts = df['dx'].value_counts()
    total = len(df)
    for cls, count in class_counts.items():
        pct = 100 * count / total
        lines.append(f"| {cls} | {count:,} | {pct:.1f}% |")
    lines.append("")

    lines.append("## 3. Distribuție per sursă")
    lines.append("")
    lines.append("| Sursă | Imagini | % | Lesion_id-uri unice |")
    lines.append("|-------|---------|---|---------------------|")
    for src in df['source'].value_counts().index:
        sub = df[df['source'] == src]
        n = len(sub)
        pct = 100 * n / total
        n_lesions = sub['lesion_id'].nunique()
        lines.append(f"| {src} | {n:,} | {pct:.1f}% | {n_lesions:,} |")
    lines.append("")

    lines.append("## 4. Distribuție clasă × sursă")
    lines.append("")
    cross = pd.crosstab(df['dx'], df['source'], margins=True, margins_name='TOTAL')
    lines.append(cross.to_markdown())
    lines.append("")

    lines.append("## 5. Mapping aplicat ISIC 2019 -> HAM10000-compatible")
    lines.append("")
    lines.append("| Original ISIC | -> | Unified | Justificare |")
    lines.append("|---------------|----|---------|-------------|")
    lines.append("| MEL | -> | mel | identitate |")
    lines.append("| NV | -> | nv | identitate |")
    lines.append("| BCC | -> | bcc | identitate |")
    lines.append("| AK | -> | akiec | AK = parte din AKIEC original |")
    lines.append("| BKL | -> | bkl | identitate |")
    lines.append("| DF | -> | df | identitate |")
    lines.append("| VASC | -> | vasc | identitate |")
    lines.append("| SCC | -> | akiec | **MERGED**: SCC + AK = AKIEC |")
    lines.append("| UNK | -> | (excluded) | clasa OOD, exclusă |")
    lines.append("")

    lines.append("## 6. Metadata pacient")
    lines.append("")
    age_avail = df['age'].notna().sum()
    sex_avail = df['sex'].notna().sum()
    site_avail = df['anatom_site'].notna().sum()
    lines.append(f"- **Vârstă disponibilă:** {age_avail:,} / {len(df):,} ({100*age_avail/len(df):.1f}%)")
    lines.append(f"- **Sex disponibil:** {sex_avail:,} / {len(df):,} ({100*sex_avail/len(df):.1f}%)")
    lines.append(f"- **Anatom_site disponibil:** {site_avail:,} / {len(df):,} ({100*site_avail/len(df):.1f}%)")
    lines.append("")

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))


report_path = os.path.join(OUTPUT_DIR, 'step0_report.md')
generate_report(df, report_path)
print(f"[SAVED] {report_path}")

[SAVED] /content/drive/MyDrive/Thesis/pipeline_outputs/step0/step0_report.md


## 10. Checklist de validare manuală

Înainte să treci la PASUL 1, verifică:

- [X] Total imagini: ~24,500-25,000 (după excluderea UNK)
- [ ] HAM10000 source: ~10,015 imagini
- [ ] BCN_20000 source: ~12,000-13,000 imagini  
- [ ] Toate cele 7 clase au reprezentare > 0
- [ ] NV e majoritar (~12-13k imagini)
- [ ] DF e minoritar (~250-300 imagini)
- [ ] Niciun lesion_id în multiple surse (verificat automat)
- [ ] Verifică câteva path-uri manual: deschide 2-3 imagini

Dacă cifrele arată ciudat (ex: HAM10000 = 0, sau toate clasele egale), e un semn că ceva nu merge cu lesion_id-urile sau cu mapping-ul. Atunci debugăm înainte să trecem la PASUL 1.

---

## Conexiune cu pașii următori

`unified_metadata.csv` e input-ul pentru:
- **PASUL 1:** Construire `test_fixed.csv` (test set fix, stratificat pe sursă × clasă)
- **PASUL 2:** Construire pool de antrenare/validare
- **PASUL 3:** Generare metadata per tier (5 niveluri de date)